# ProxyTrace 01. 위임장 수집 (SEC EDGAR API)

**티커 하나만 넣으면** 그 기업의 최신 위임장(DEF 14A)과 직전 위임장을 받아요.

- 최신: 분석 대상
- 직전: 작년 반론과 비교하기 위한 자료

결과
- `data/raw/{TICKER}/{제출일}.htm`
- `data/raw/{TICKER}/manifest.json` ← 02 노트북이 이 파일을 읽어요

In [ ]:
!pip install requests -q

## 1. 설정

In [ ]:
import json
import os
import time
from datetime import date
from pathlib import Path

import requests

# ▶ 여기만 바꾸면 어떤 기업이든 돌아가요
TICKER = "META"

N_PROXIES = 2          # 최신 위임장 + 직전 위임장 (작년 비교용)
MIN_GAP_DAYS = 300     # 같은 해에 임시주총 위임장이 또 나온 경우를 걸러내기 위한 최소 간격

# SEC는 User-Agent(이름+이메일)가 없는 요청을 막아요.
# 환경변수 SEC_USER_AGENT가 있으면 그 값을, 없으면 아래 기본값을 써요.
HEADERS = {"User-Agent": os.environ.get("SEC_USER_AGENT", "YOUR_NAME your_email@example.com")}
assert "YOUR_NAME" not in HEADERS["User-Agent"], "SEC_USER_AGENT를 설정하거나 기본값을 본인 정보로 바꾸세요"

FORMS = {"DEF 14A"}    # 정식 위임장만 (DEFA14A 추가 자료, DEFM14A 합병 위임장은 제외)
RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

## 2. 함수 정의

In [ ]:
def get(url: str) -> requests.Response:
    """SEC 요청 + 초당 10회 제한을 지키기 위한 대기"""
    r = requests.get(url, headers=HEADERS, timeout=60)
    r.raise_for_status()
    time.sleep(0.2)
    return r


def ticker_to_cik(ticker: str) -> tuple[str, str]:
    """티커 → (CIK, 회사명). SEC가 공개하는 티커 목록 파일을 써요."""
    data = get("https://www.sec.gov/files/company_tickers.json").json()
    for row in data.values():
        if row["ticker"].upper() == ticker.upper():
            return str(row["cik_str"]), row["title"]
    raise ValueError(f"티커 {ticker}를 SEC 목록에서 찾지 못했어요")


def collect_filings(cik: str) -> list[dict]:
    """공시 목록 전체. recent에 없는 과거 공시는 files 페이지에 있어요."""
    sub = get(f"https://data.sec.gov/submissions/CIK{cik.zfill(10)}.json").json()

    def to_rows(block: dict) -> list[dict]:
        keys = ["form", "filingDate", "accessionNumber", "primaryDocument"]
        return [dict(zip(keys, vals)) for vals in zip(*(block[k] for k in keys))]

    rows = to_rows(sub["filings"]["recent"])
    for f in sub["filings"].get("files", []):
        rows += to_rows(get(f"https://data.sec.gov/submissions/{f['name']}").json())
    return rows


def pick_proxies(rows: list[dict], n: int = N_PROXIES, min_gap: int = MIN_GAP_DAYS) -> list[dict]:
    """최신 위임장부터 n개 고르기.
    연도로 고르지 않는 이유: 회사마다 위임장 내는 달이 달라요 (Meta 4월, Apple 1월, Microsoft 10월).
    직전 것과 min_gap일 이상 떨어진 것만 골라서 임시주총 위임장을 걸러요."""
    cands = sorted((r for r in rows if r["form"] in FORMS), key=lambda r: r["filingDate"], reverse=True)
    picked = []
    for r in cands:
        d = date.fromisoformat(r["filingDate"])
        if all((date.fromisoformat(p["filingDate"]) - d).days >= min_gap for p in picked):
            picked.append(r)
        if len(picked) == n:
            break
    return picked


def download(row: dict, cik: str, save_path: Path) -> str:
    acc = row["accessionNumber"].replace("-", "")
    url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc}/{row['primaryDocument']}"
    save_path.write_bytes(get(url).content)
    return url

## 3. 티커 → CIK

In [ ]:
CIK, COMPANY = ticker_to_cik(TICKER)
print(f"{TICKER} → CIK {CIK} ({COMPANY})")

## 4. 위임장 고르기
다운로드 전에 어떤 문서를 고르는지 먼저 확인해요.

In [ ]:
rows = collect_filings(CIK)
proxies = pick_proxies(rows)
print(f"공시 {len(rows)}건 중 위임장 {len(proxies)}개 선택")
for i, p in enumerate(proxies):
    print(f"  [{'최신' if i == 0 else '직전'}] {p['filingDate']}  {p['accessionNumber']}")

## 5. 다운로드 + 기록 저장

In [ ]:
out_dir = RAW_DIR / TICKER.lower()
out_dir.mkdir(parents=True, exist_ok=True)

manifest = {"ticker": TICKER, "cik": CIK, "company": COMPANY, "proxies": []}
for i, p in enumerate(proxies):
    path = out_dir / f"{p['filingDate']}.htm"
    url = download(p, CIK, path)
    print(f"{p['filingDate']} → {path} ({path.stat().st_size // 1024} KB)")
    manifest["proxies"].append({**p, "role": "latest" if i == 0 else "previous",
                                "url": url, "saved_to": str(path)})

(out_dir / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
manifest

## 다음 단계
`02_clean_text.ipynb`에서 같은 `TICKER`로 실행하세요. 다른 기업으로 테스트하려면 1번 셀의 `TICKER`만 바꾸면 돼요 (예: `"WMT"`, `"MSFT"`, `"AAPL"`).